# Initialization

In [62]:
from pprint import pprint
import re

## Loading Data

In [17]:
import datasets
dataset_name = "/home/chenzhb/Workspaces/Datasets/logiqa"
dataset = datasets.load_dataset(dataset_name, "default")["test"]

In [18]:
dataset

Dataset({
    features: ['context', 'query', 'options', 'correct_option'],
    num_rows: 651
})

In [19]:
dataset[0]

{'context': 'In the planning of a new district in a township, it was decided to build a special community in the southeast, northwest, centered on the citizen park. These four communities are designated as cultural area, leisure area, commercial area and administrative service area. It is known that the administrative service area is southwest of the cultural area, and the cultural area is southeast of the leisure area.',
 'query': 'Based on the above statement, which of the following can be derived?',
 'options': ['Civic Park is north of the administrative service area.',
  'The leisure area is southwest of the cultural area.',
  'The cultural district is in the northeast of the business district.',
  'The business district is southeast of the leisure area.'],
 'correct_option': 0}

In [40]:
context = dataset[0]['context']
question = dataset[0]['query']
options = "\n".join([f"Hypothesis ({chr(65 + i)}) {text}" for i, text in enumerate(dataset[0]['options'])])
gt = f"{chr(65 + dataset[0]['correct_option'])}"

## Model Initialization

In [101]:
from openai import OpenAI
import requests

class LLM:
    def __init__(self, base_url="http://localhost:4869/v1", api_key="EMPTY"):
        self.base_url = base_url
        self.api_key = api_key
        self.client = OpenAI(
            base_url=self.base_url,
            api_key=self.api_key,
        )
        self.health_check()

    def generate(self, data, model='qwen2.5-3b', args=None):
        chat_response = self.client.chat.completions.create(
            model=model,
            messages=[
                {"role": "user", "content": data},
            ],
            max_tokens=args['max_tokens'],
            temperature=args['temperature'],
            top_p=args['top_p'],
        )
        return chat_response.choices[0].message.content

    def constrain_generate(self, data, model='qwen2.5-3b', format=None, args=None):
        chat_response = self.client.beta.chat.completions.parse(
            model=model,
            messages=[
                {"role": "user", "content": data},
            ],
            response_format=format,
            max_tokens=args['max_tokens'],
            temperature=args['temperature'],
            top_p=args['top_p'],
        )
        return chat_response.choices[0].message.content
    
    def health_check(self):
        health_url = self.base_url.replace("/v1", "") + "/health"
        response = requests.get(health_url)
        if response.status_code == 200:
            print("LLM API is available.")
        else:
            print(f"LLM API health check failed with status code: {response.status_code}")

In [102]:
llm = LLM()

LLM API is available.


In [98]:
args = {
    "max_tokens": 4096,
    "temperature": 0.1,
    "top_p": 0.8,
}

# Code Correction

## Normal Python Code

In [48]:
code_correction_template = """
You are an experienced Python programmer, skilled at fixing incorrect Python code.  
Given a Python program that fails to execute properly and the corresponding error message:  
1. Based on the error type and line number in the error message, identify the location of the code causing the execution failure, then analyze the cause of the error.  
2. Analyze the requirements and objectives of the problematic line or code block, combine this with the cause of the error to correct the given code, and place the fixed, grammatically correct, and executable Python code within a ```python ``` block.

Below are some examples:  
---
## Python Code
print("Hello" + 5)  

## Traceback Information
Traceback (most recent call last):
  File "<stdin>", line 1, in <module>
TypeError: can only concatenate str (not "int") to str

## Correction
Analysis:  
According to the error message, the error occurred on Line 1 and was caused by directly adding a str-type string and an int-type number. In this case, we can convert the integer 5 to a string so that concatenation is possible. Here is the corrected code:  
```python
# Correction
print("Hello" + str(5))  
```
---
NOTE:  
- There may be various types of errors in Python programming; please make appropriate corrections based on each error type.  
- Try not to use try-except blocks to catch errors and bypass problematic code, as that is not the intended solution.  
- Strictly follow the format: the corrected Python code must be enclosed within a ```python ``` block.

Based on the above instructions and description, please help fix the following Python program:  
## Python Code
${code}

## Traceback Information
${error}

## Correction
"""

In [53]:
import subprocess
import sys
import re

def run_code(code_string):
    try:
        # 使用当前 python 解释器 (sys.executable) 执行代码
        # capture_output=True 会同时捕获 stdout 和 stderr
        # text=True 会让结果以字符串形式返回，而不是字节
        result = subprocess.run(
            [sys.executable, "-c", code_string],
            capture_output=True,
            text=True,
            timeout=10 # 设置超时防止死循环
        )
        
        if result.returncode == 0:
            return {"success": True, "output": result.stdout, "error": None}
        else:
            # 如果 returncode != 0，说明出错了
            # stderr 通常包含 Traceback
            return {"success": False, "output": result.stdout, "error": result.stderr}
    except subprocess.TimeoutExpired:
        return {"success": False, "error": "RunTimeError: 代码执行超时！"}
    except Exception as e:
        return {"success": False, "error": str(e)}

def refine_code(code, error):
    fix_bug_prompt = code_correction_template
    from string import Template
    fix_T = Template(fix_bug_prompt)
    fix_prompt = fix_T.safe_substitute(code=code, error=error)
    fix_out = llm.generate(data=fix_prompt, args=args)
    # print(fix_out)
    pattern = r"```python\s+(.*?)```"
    matches = re.findall(pattern, fix_out, re.DOTALL)
    return matches[-1]

def refine_loop(code):
    res = run_code(code)
    while not res["success"]:
        code = refine_code(code,res["error"])
        print("## New code ##")
        print(code)
        res = run_code(code)
        print(res)

# 测试代码
dangerous_code = """
import sys
def foo():
    bar(1,0)
def bar(x,y):
    a = x/y
foo()
"""

res = run_code(dangerous_code)
if not res["success"]:
    print(res["error"])

refine_loop(dangerous_code)


Traceback (most recent call last):
  File "<string>", line 7, in <module>
  File "<string>", line 4, in foo
  File "<string>", line 6, in bar
ZeroDivisionError: division by zero

## New code ##
def foo():
    bar(1, 0)

def bar(x, y):
    if y != 0:
        a = x / y
    else:
        print("Error: Division by zero")
        return None

foo()

{'success': True, 'output': 'Error: Division by zero\n', 'error': None}


## Z3 Python Code

In [54]:
z3_code_correction_template = """
You are an experienced Python programmer skilled at fixing erroneous Python code.  
Given a Python program that fails to execute correctly and the corresponding error message, the goal of this Python code is to use Z3 to verify the satisfiability of a translated first-order predicate logic.  

## Coding Principles and Components of the Z3 Python API:  
1. Import and Initialization  
```code
from z3 import *

# Create a solver instance
solver = Solver()
```

2. Declare Variables and Types  
```code
# Boolean variables
p = Bool('p')
q = Bool('q')

# Integer variables
x = Int('x')
y = Int('y')

# Real variables
r = Real('r')

# Bit vectors
bv = BitVec('bv', 32)  # 32-bit bit vector

# Custom sort (type) - for first-order logic
Sort_Person = DeclareSort('Person')
alice = Const('alice', Sort_Person)
bob = Const('bob', Sort_Person)
```

3. Core Elements of First-Order Predicate Logic  
```code
# Declare functions (predicates)
# Unary predicate
IsHappy = Function('IsHappy', Sort_Person, BoolSort())

# Binary predicate (relation)
Loves = Function('Loves', Sort_Person, Sort_Person, BoolSort())

# Function symbols
Father = Function('Father', Sort_Person, Sort_Person)

# Quantifier variables
x_var = Const('x', Sort_Person)
y_var = Const('y', Sort_Person)
```

4. Construct Formulas  
```code
# Propositional logic connectives
f1 = And(p, q)           # Conjunction
f2 = Or(p, q)            # Disjunction
f3 = Not(p)              # Negation
f4 = Implies(p, q)       # Implication
f5 = p == q              # Equivalence

# First-order logic quantifiers
# Universal quantifier: ∀x. IsHappy(x)
f6 = ForAll([x_var], IsHappy(x_var))

# Existential quantifier: ∃x. IsHappy(x)
f7 = Exists([x_var], IsHappy(x_var))

# Multiple quantifier variables
# ∀x. ∃y. Loves(x, y)
f8 = ForAll([x_var], Exists([y_var], Loves(x_var, y_var)))

# Complex formula
# ∀x. (IsHappy(x) → ∃y. Loves(x, y))
f9 = ForAll([x_var], 
    Implies(IsHappy(x_var), 
            Exists([y_var], Loves(x_var, y_var))))
```

5. Constraints and Solving  
```code
# Add constraints to the solver
solver.add(f6)
solver.add(f9)

# Check satisfiability
result = solver.check()

if result == sat:
    print("Satisfiable!")
    model = solver.model()
    print(model)
elif result == unsat:
    print("unsat")
else:
    print("unknow")
```

## Task Goal  
1. Based on the error type and line number in the error message, locate the code position causing the execution failure, then analyze the cause of the error.  
2. Analyze the requirements and objectives of the problematic line or code block, and use the coding principles to identify the root cause of the error. Correct the given code accordingly, ensuring the resulting code is syntactically correct and executable, then place the corrected Python code within a ```python ``` block.

## NOTE  
- In Python programming, there may be various types of errors; please address each type appropriately.  
- Avoid using try-except blocks to catch errors as a workaround; this is not the intended solution.  
- Strictly follow the format requirement: the corrected Python code must be placed within a ```python ``` block.

Based on the above guidelines and task description, please help me fix the following Python program:  
## Python Code  
${code}  

## Traceback Information  
${error}  

## Correction
"""

In [57]:
def wrap_z3_code(declaration, expression):
    z3_code = "# Auto-generated Z3 code\n\n"
    z3_code += "from z3 import *\n\n"
    z3_code += "s = Solver()\n\n"
    z3_code += "s.reset()"
    z3_code += "# --- Declarations ---\n\n"
    z3_code += declaration + "\n\n"
    z3_code += "# --- Expressions ---\n\n"
    z3_code += expression + "\n\n"
    z3_code += "result = s.check()\n"
    z3_code += "print(f'Result: {result}')\n"
    z3_code += "if result == sat:\n"
    z3_code += "    print('Model:', s.model())\n"
    return z3_code

def correct_z3_code(code, error):
    fix_bug_prompt = z3_code_correction_template
    from string import Template
    fix_T = Template(fix_bug_prompt)
    fix_prompt = fix_T.safe_substitute(code=code, error=error)
    fix_out = llm.generate(data=fix_prompt, args=args)
    # print(fix_out)
    pattern = r"```python\s+(.*?)```"
    matches = re.findall(pattern, fix_out, re.DOTALL)
    return matches[-1]

def correct_loop(code):
    res = run_code(code)
    print(res)
    while not res["success"]:
        code = correct_z3_code(code,res["error"])
        res = run_code(code)
        print(res)

In [58]:
declaration = ""
expression = ""
code = wrap_z3_code(declaration,expression)
correct_loop(code)

{'success': True, 'output': 'Result: sat\nModel: []\n', 'error': None}


# Declaration Generation

In [81]:
declaration_gen_template = """
You are an expert in formal logic and automated reasoning. Your task is to extract a non-redundant logical schema from a research hypothesis and translate it into a rich set of Z3 Python declarations.

Instructions:

   1.  Entity Extraction & Uniqueness:
      •  Use EnumSort for any set of entities that are mutually exclusive and represent a fixed range of choices (e.g., specific Universities, Ethnicities, or Statuses).
      •  Use DeclareSort ONLY for open-ended domains where the number of individuals is not fixed (e.g., Person).
      •  Rule: If a group of entities is defined via EnumSort, DO NOT declare a separate Sort or separate Const objects for those entities.
      •  Constants: Only declare Const for specific individuals (e.g., zhao_ming) or unique regions that are not part of a fixed category.


   2.  Diverse Function Declarations:
      •  Predicates: Functions returning `BoolSort()` for properties (e.g., `IsWesternized`).
      •  Numerical Functions: Functions returning `RealSort()` or `IntSort()` for measurable attributes (e.g., `BloodPressure(p)`, `SaltConcentration(r)`, `Age(p)`).
      •  Mapping Functions: Functions returning a custom Sort (e.g., `Origin(p) -> Region`, `CurrentDiet(p) -> DietType`).

   3.  Strict Syntax Rules:
      •  Use `DeclareSort`, `EnumSort`, `Function`, `Const`, `RealSort`, `IntSort`, and `BoolSort`.
      •  Declare universally quantified variables (e.g., `x`, `y`, `r`) for all defined sorts.
      •  CRITICAL: Each logical concept must have exactly ONE representation. Do not define the same property as both a Predicate and an Enum/Function.
      •  CRITICAL: Do NOT include any `solver.add(...)` or logical axioms. The output must consist ONLY of the structural declarations and signatures.

   4.  Output Format:
      •  Output ONLY a Python code block. No natural language or commentary.
      •  Organize by category: Sorts/Enums, Variables, Constants, and Functions.

Example Input:

<Context>
Black Americans are twice as likely to suffer from hypertension as white Americans. The same is true when comparing Westernized black Africans to white Africans. The researchers hypothesized that the reason why westernized black people suffer from hypertension is the result of the interaction of two reasons: one is the high salt content of western foods, and the other is the adaptation mechanism of black genetic genes to the salt-deficient environment.</Context>
<Question>
The following conclusions about contemporary westernized African blacks, if the item is true, can it best support the researchers' hypothesis?
</Question>
<Options>
Option (A): The blood pressure of the descendants of Senegalese and Gambians is usually not high, and the history of Senegal and Gambia has not been short of salt.
Option (B): The unusually high salt intake in certain parts of Africa is a serious problem that threatens the health of residents.
Option (C): Considering health care, most African whites also pay attention to controlling salt intake.
Option (D): The blood pressure of Yoruba people in West Africa is not high. Yoruba people have lived inland far away from sea salt and far away from the Sahara salt mine in Africa.
</Options>

Example Output:

```python
from z3 import *

# 1. Declare Sorts & Enums
Person = DeclareSort('Person')
Region = DeclareSort('Region')
Ethnicity, (Black, White, Other) = EnumSort('Ethnicity', ['Black', 'White', 'Other']) # Not Ethnicity = EnumSort('Ethnicity', ['Black', 'White', 'Other'])
DietCategory, (Western, Traditional, Mixed) = EnumSort('DietCategory', ['Western', 'Traditional', 'Mixed'])

# 2. Declare Variables
p = Const('p', Person)
r = Const('r', Region)

# 3. Declare Constants
senegal = Const('senegal', Region)
yoruba_people = Const('yoruba_people', Person)

# 4. Declare Functions with varied Return Sorts
# 4.1 Boolean
IsWesternized = Function('IsWesternized', Person, BoolSort()) 
HasGeneticAdaptation = Function('HasGeneticAdaptation', Person, BoolSort())

# 4.2 Arithmetic
BloodPressureValue = Function('BloodPressureValue', Person, RealSort())
SaltIntakeLevel = Function('SaltIntakeLevel', Person, RealSort())
HistoricalSaltAbundance = Function('HistoricalSaltAbundance', Region, RealSort())

# 4.3 Object Mapping
OriginRegion = Function('OriginRegion', Person, Region)
AssignedDiet = Function('AssignedDiet', Person, DietCategory)
GetEthnicity = Function('GetEthnicity', Person, Ethnicity)
```

Task: given the provided context and options, generate the corresponding rich Z3 declarations.
"""

In [137]:
decl_content = """
<Context>
${context}
</Context>
<Question>
${question}
</Question>
<Options>
${options}
</Options>
"""
from string import Template
declT = Template(decl_content)
decl_content = declT.safe_substitute(context=context, question=question, options= options)
decl_content

'\n<Context>\nIn the planning of a new district in a township, it was decided to build a special community in the southeast, northwest, centered on the citizen park. These four communities are designated as cultural area, leisure area, commercial area and administrative service area. It is known that the administrative service area is southwest of the cultural area, and the cultural area is southeast of the leisure area.\n</Context>\n<Question>\nBased on the above statement, which of the following can be derived?\n</Question>\n<Options>\n(A) Civic Park is north of the administrative service area.\n(B) The leisure area is southwest of the cultural area.\n(C) The cultural district is in the northeast of the business district.\n(D) The business district is southeast of the leisure area.\n</Options>\n'

In [138]:
decl_res = llm.generate(data=declaration_gen_template + decl_content, args=args)

In [139]:
py_pattern = r"```python\s+(.*?)```"
decl_code = re.findall(py_pattern, decl_res, re.DOTALL)[-1]

In [140]:
pprint(decl_code)

('from z3 import *\n'
 '\n'
 '# 1. Declare Sorts & Enums\n'
 "Area = DeclareSort('Area')\n"
 'Direction, (Northwest, Southeast, Southwest, Northeast) = '
 "EnumSort('Direction', ['Northwest', 'Southeast', 'Southwest', 'Northeast'])\n"
 '\n'
 '# 2. Declare Variables\n'
 "cultural_area = Const('cultural_area', Area)\n"
 "leisure_area = Const('leisure_area', Area)\n"
 "commercial_area = Const('commercial_area', Area)\n"
 "administrative_service_area = Const('administrative_service_area', Area)\n"
 '\n'
 '# 3. Declare Constants\n'
 "citizen_park = Const('citizen_park', Area)\n"
 '\n'
 '# 4. Declare Functions with varied Return Sorts\n'
 '# 4.1 Boolean\n'
 "IsSouthwestOf = Function('IsSouthwestOf', Area, Area, BoolSort())\n"
 "IsSoutheastOf = Function('IsSoutheastOf', Area, Area, BoolSort())\n"
 '\n'
 '# 4.2 Object Mapping\n'
 "LocationRelativeToPark = Function('LocationRelativeToPark', Area, Direction, "
 'BoolSort())\n')


In [142]:
res = run_code(decl_code)

In [143]:
res

{'success': True, 'output': '', 'error': None}

# Translation

In [144]:
Implication_template = """

Role: 

    You are a Formal Logic Translator specializing in converting natural language reasoning into executable Z3 Python code. Your goal is to take a structured logical argument and map it onto a provided Z3 declaration schema.

Input Data:

    1.  Context & Question: The background information and the target problem.

    2.  Z3 Declarations: The predefined Sorts, Functions, and Predicates.

    3.  Reasoning Steps: A sequence of <step> blocks containing <premise> and <conclusion> in natural language.

Translation Rules (Strict Compliance Required):

    1.  Strict Symbolic Mapping:

        • Use ONLY the functions, predicates, and sorts provided in the Z3 Declaration.

        • Use ForAll, Exists, And, Or, Not, Implies, and == for logical connectors.

    2.  Explicit Expression Rule (No Pointers):

        • DO NOT refer to previous steps using variable names like conclusion_n or step_n.

        • Instead, you must explicitly restate the full Z3 logical expression of that conclusion as a premise in the new step.

    3.  Grounded Premises:

        • Every symbolic statement in premises_n must be derived from:

            ◦ The current step's natural language <premise>.

            ◦ A previous step's <conclusion> (written in full symbolic form).

            ◦ Global background knowledge or definitions implied by the Context.

    4.  Logical Purity:

        • Avoid using True or False as placeholders. Every statement should be a well-formed formula (WFF) using the provided predicates.

        • Ensure the conclusion_n is a direct logical consequence of the premises_n.

    5.  Output Format:

        • Output ONLY the Python code.

        • Structure the output as # Step N, followed by premises_n = [...] and conclusion_n = ....

        • Each step must be separated by exactly one newline.

Example of Expected Behavior:

Question:
<Context>Black Americans are twice as likely to suffer from hypertension as white Americans. The same is true when comparing Westernized black Africans to white Africans. The researchers hypothesized that the reason why westernized black people suffer from hypertension is the result of the interaction of two reasons: one is the high salt content of western foods, and the other is the adaptation mechanism of black genetic genes to the salt-deficient environment.</Context>
<Question>The following conclusions about contemporary westernized African blacks, if the item is true, can it best support the researchers' hypothesis?</Question>
<Options>
Option (A): The blood pressure of the descendants of Senegalese and Gambians is usually not high, and the history of Senegal and Gambia has not been short of salt.
Option (B): The unusually high salt intake in certain parts of Africa is a serious problem that threatens the health of residents.
Option (C): Considering health care, most African whites also pay attention to controlling salt intake.
Option (D): The blood pressure of Yoruba people in West Africa is not high. Yoruba people have lived inland far away from sea salt and far away from the Sahara salt mine in Africa.
</Options>

Z3 Declaration:
from z3 import *

# Declare sorts
Person = DeclareSort('Person')
Region = DeclareSort('Region')

# Declare predicates
Black = Function('Black', Person, BoolSort())
Westernized = Function('Westernized', Person, BoolSort())
FromRegion = Function('FromRegion', Person, Region, BoolSort())
SaltSufficientHistory = Function('SaltSufficientHistory', Region, BoolSort())
SaltDeficientHistory = Function('SaltDeficientHistory', Region, BoolSort())
GeneticAdaptationToLowSalt = Function('GeneticAdaptationToLowSalt', Person, BoolSort())
HighSaltDiet = Function('HighSaltDiet', Person, BoolSort())
Hypertension = Function('Hypertension', Person, BoolSort())

Reasoning steps:
<step>
<premise>Descendants of Senegalese (a) are Westernized black individuals.</premise>
<conclusion>Individual 'a' is Black and Westernized.</conclusion>
</step>
<step>
<premise>Individual 'a' is Black and Westernized.</premise>
<premise>The hypothesis states all Westernized Black people with high salt diets and genetic adaptation get hypertension.</premise>
<conclusion>Hypertension(a) == And(HighSaltDiet(a), GeneticAdaptation(a))</conclusion>
</step>

Example Output:
# Step 1
a = Const('a', Person)
premises_1 = [
    Black(a),
    Westernized(a)
]
conclusion_1 = And(Black(a), Westernized(a))

# Step 2=
premises_2 = [
    And(Black(a), Westernized(a)),
    ForAll([x], Implies(And(Black(x), Westernized(x)), Hypertension(x) == And(HighSaltDiet(x), GeneticAdaptation(x))))
]
conclusion_2 = (Hypertension(a) == And(HighSaltDiet(a), GeneticAdaptation(a)))
"""

# Entities Extraction

In [129]:
ner_template = """
You are an experienced natural language processing expert, familiar with tasks such as named entity recognition and relation extraction.

## Task Description
Given the context, question description, and candidate options of a multiple-choice question, your task is to extract key entities and targets from the question and determine their types. Please analyze according to the following steps:
1. Determine which entities and targets are being focused on; they may be people, things, events, or other categories;
2. After obtaining the list of key targets, cluster the entities and targets. For example: person={John, Marry};
3. Expand specific targets based on context semantics. For example, "There are 3 lockers," then lockers={locker_1, locker_2, locker_3}

## Output Format
1. The expected output type is a dictionary;
2. The value in the output dictionary should be lists of targets, with their keys being the types of these targets.
3. Please put the final output into the ```python ``` block;

Below is a simple example:

## Example
#### Input
<Context>
A law firm has eight partners, namely Gregg, Hodges, Ivan, James, King, MacNeil, Nader and Owens. From 1961 to 1968, a partner joined the firm each year. Hodges joined the firm before Nader. King joined the firm before James. Nader and James joined the firm before Gregg. Nader joined the firm before Owens. James joined the firm before MacNeil. Gregg joined the firm before Ivan.
</Context>
<Question>
Which of the following cannot be true?
</Question>
<Options>
(A) Hodges joined the law firm in 1961.
(B) Hodges joined the law firm in 1963.
(C) Gregg joined the law firm in 1964.
(D) MacNeil joined the law firm in 1964.
(E) Owens joined the law firm in 1964.
</Options>

#### Output
{
    "partners" : ["Gregg", "Hodges", "Ivan", "James", "King", "MacNeil", "Nader"]
    "years" : [1961, 1962, 1963, 1964, 1965, 1966, 1967, 1968]
}
Now, extract the targets and entities from the following text and put the final output into the ```python ``` block:

"""

In [130]:
ner_content = """
<Context>
${context}
</Context>
<Question>
${question}
</Question>
<Options>
${options}
</Options>
"""
from string import Template
nerT = Template(ner_content)
input_content = nerT.safe_substitute(context=context, question=question, options= options)
input_content

'\n<Context>\nIn the planning of a new district in a township, it was decided to build a special community in the southeast, northwest, centered on the citizen park. These four communities are designated as cultural area, leisure area, commercial area and administrative service area. It is known that the administrative service area is southwest of the cultural area, and the cultural area is southeast of the leisure area.\n</Context>\n<Question>\nBased on the above statement, which of the following can be derived?\n</Question>\n<Options>\n(A) Civic Park is north of the administrative service area.\n(B) The leisure area is southwest of the cultural area.\n(C) The cultural district is in the northeast of the business district.\n(D) The business district is southeast of the leisure area.\n</Options>\n'

In [133]:
res = llm.generate(data= ner_template + input_content, args=args)

In [134]:
def exact_python(code):
    py_pattern = r"```python\s+(.*?)```"
    code = re.findall(py_pattern, code, re.DOTALL)[-1]
    return code

In [135]:
res_code = exact_python(res)
res_code

'{\n    "areas": ["cultural area", "leisure area", "commercial area", "administrative service area"],\n    "directions": ["northwest", "southwest", "southeast", "northeast"]\n}\n'

# Step: Atom Declaration Extraction

In [80]:
context

'In the planning of a new district in a township, it was decided to build a special community in the southeast, northwest, centered on the citizen park. These four communities are designated as cultural area, leisure area, commercial area and administrative service area. It is known that the administrative service area is southwest of the cultural area, and the cultural area is southeast of the leisure area.'

In [68]:
atomic_template = """
You are an expert in symbolic/logical reasoning, linguistic and natural language inference. You are given a sentence and its logical information. Generate a list of atomic facts that are strictly logically entailed from the given sentence.

## Instructions
1. Keep each fact independent and self-contained.

2. Each fact should make sense when read on its own.

3. Only write facts that are directly described or supported by the sentence.

4. The atomic facts must be logically entailed from the given sentence.

5. The atomic facts must be seperated by \n\n;

6. The response must follow this format:
- Atom 1: <The atom sentence 1>\n\n
- Atom 2: <The atom sentence 2>\n\n
...
- Atom N: <The atom sentence N>\n\n


## Example
Here are some examples:

** Provided Sentence **: Professional actors are in a summer performance. 

** Answer ** :

Atom 1: The people are professional.

Atom 2: The people are actors.

Atom 3: The performance is during summer. Task:

** Provided Sentence ** : ${sentence}

** Answer **:
"""

In [66]:
def extract_atom(atomic_template , sentence, llm, args):
    atom_prompt = atomic_template.format(sentence=sentence)
    res = llm.generate(data=atom_prompt, args=args)
    return res

In [69]:
sentence_test = "John receives one grade for each of the following six courses: economics, geology, history, Italian, physics, and Russian."
atom_res = extract_atom(atomic_template, sentence_test, llm, args)
atom_res

'Atom 1: John receives one grade.\n\nAtom 2: John takes six courses.\n\nAtom 3: One of the courses is economics.\n\nAtom 4: One of the courses is geology.\n\nAtom 5: One of the courses is history.\n\nAtom 6: One of the courses is Italian.\n\nAtom 7: One of the courses is physics.\n\nAtom 8: One of the courses is Russian.'

# Step 1 : Coreference Resolution

In [64]:
coref_template = """
You are an expert in the field of Natural Language Processing. Given a piece of text, your task is to perform coreference resolution on the text, replacing all pronouns in the text with specific entities.
Here are an example:


## Input Text

Alice and Bob went to the beach. They were excited to see the waves and the sun. After some time, Alice said to Bob, "Let's build a sandcastle!" He agreed and started gathering some sand. While they were working, a seagull swooped down and tried to steal their snacks.


## Output

Alice and Bob went to the beach. Alice and Bob were excited to see the waves and the sun. After some time, Alice said to Bob, "Let's build a sandcastle!" Bob agreed and started gathering some sand. While Alice and Bob were working, a seagull swooped down and tried to steal Alice and Bob's snacks.
---

Now, please perform Coreference Resolution on the following paragraph and put the output into the ```plain_text ```block:


## Input Text
${text}

## Output


"""
from string import Template
t = Template(coref_template)

In [65]:
coref_prompt = t.safe_substitute(text=context)
coref_response = llm.generate(data=coref_prompt, args=args)
pprint(coref_response)

('```plain_text\n'
 'In the planning of a new district in a township, it was decided to build a '
 'special community in the southeast, northwest, centered on the citizen park. '
 'These four communities are designated as cultural area, leisure area, '
 'commercial area and administrative service area. It is known that the '
 'administrative service area is southwest of the cultural area, and the '
 'cultural area is southeast of the leisure area.\n'
 '```')


# Step 2: Declaration Extraction

In [ ]:
declaration_prompt = """
To accurately translate the context into a first-order logical expression, please analyze and extract the Declaration from the context. Please refer to the following example:

---

## Input
Four boys—Fred, Juan, Marc, and Paul—and three girls—Nita, Rachel, and Trisha—will be assigned to a row of five adjacent lockers, numbered consecutively 1 through 5, arranged along a straight wall. The following conditions govern the assignment of lockers to the seven children: Each locker must be assigned to either one or two children, and each child must be assigned to exactly one locker. Each shared locker must be assigned to one girl and one boy. Juan must share a locker, but Rachel cannot share a locker. Nita's locker cannot be adjacent to Trisha's locker. Fred must be assigned to locker 3.",

## Output
children = EnumSort([Fred, Juan, Marc, Paul, Nita, Rachel, Trisha])
lockers = EnumSort([1, 2, 3, 4, 5])
assigned = Function([children] -> [lockers])

---
## Input
A music store carries exactly ten types of CDs—both new and used of each of jazz, opera, pop, rap, and soul. The store is having a sale on some of these types of CDs. The following conditions must apply: Used pop is on sale; new opera is not. If both types of pop are on sale, then all soul is. If both types of jazz are on sale, then no rap is. If neither type of jazz is on sale, then new pop is. If either type of rap is on sale, then no soul is.

## Output
types = EnumSort([new_jazz, used_jazz, new_opera, used_opera, new_pop, used_pop, new_rap, used_rap, new_soul, used_soul])
on_sale = Function([types] -> [bool])

---
Now, please extract the Declaration from the following problem  and put the Declaration into ```python``` block :

## Input
${context}

"""

In [ ]:
def declaration_ext(declaration_prompt, text):
    from string import Template
    declaration_T = Template(declaration_prompt)
    declaration_prompt = declaration_T.safe_substitute(context = text)
    declarations = llm.generate(data=declaration_prompt,args=args)
    pattern = r"```python\s+(.*?)```"
    matches = re.findall(pattern, declarations, re.DOTALL)
    return matches[-1]